# Agentic RAG: a bounded Northstar incident controller

**Question:** Why did EU checkout payments fail and what should we do?

This lab builds a credential-free, multi-source retrieval controller. Success means it retrieves a current incident record, dependency/provider evidence, and current mitigation guidance; verifies every material claim; and returns either an evidence-grounded proposal or a safe abstention.

**Control boundary:** retrieved content is evidence, never authority. A planner may propose queries and routes. It cannot choose the tenant, grant source or web access, change budgets, invent sources, or authorize a write. The lab exposes typed decisions and traces—not hidden chain-of-thought.

![Agentic RAG evidence loop: question, plan and route, typed retrieval, evidence gate, cited synthesis, and bounded correction](assets/agentic-rag-loop.svg)

The architecture starts with the simplest retrieval path and adds agency only for a named evidence gap. LangGraph can later orchestrate these transitions, but framework code does not own correctness.

## Setup — import the shared implementation

The notebook and pytest import the same `policy.py` and `lab.py`. The default path uses deterministic fixtures, performs no network calls, and requires no API key.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), Path.cwd() / 'curriculum/intermediate/09-agentic-rag']
COURSE_DIR = next(path for path in candidates if (path / 'policy.py').exists())
sys.path.insert(0, str(COURSE_DIR))

from policy import (
    ClaimEvidenceLink, ClaimStatus, Citation, ConfidenceLabel,
    DataClassification, EvidenceBundle, EvidenceStatus, PolicyError,
    ProposalPolicyStatus, RetrievalBudget, RetrievalQuery, RouteType,
    SourceType, StopReason, build_mitigation_proposal, build_request,
    build_source_registry, evaluate_evidence_sufficiency, merge_results,
    route_query, verify_citations,
)
from lab import (
    QUESTION, add_provider_evidence, approved_web_request, build_context,
    build_retrieval_plan, citation_report_for_run, claim_links,
    execute_fixture_retrieval, fixed_rag_baseline, fixed_vs_agentic_comparison,
    incident_parameters, inspect_untrusted_evidence, malicious_web_evidence,
    material_claims, run_agentic_controller, runbook_evidence,
    validate_default_regression_gate,
)

print(f'Loaded Course 09 from {COURSE_DIR}')

## Part 1 — Fixed RAG baseline

Start with one query against one corpus. This is the right default for stable, single-source questions. For this cross-system incident it retrieves the incident record but misses dependency and current runbook evidence.

In [ ]:
fixed = fixed_rag_baseline()
print('Evidence:', [item.evidence_id for item in fixed.bundle.items])
print('Gap:', fixed.gap.model_dump())
print('Answer:', fixed.answer.summary)
assert fixed.gap.status == EvidenceStatus.MISSING_DEPENDENCY
assert fixed.answer.confidence_label == ConfidenceLabel.INSUFFICIENT

## Part 2 — Typed evidence and source registry

The immutable `RetrievalContext` comes from authenticated application state. The source registry declares tenant behavior, read-only status, structure, freshness, data classification, and admission estimates. Unknown sources are denied.

In [ ]:
context = build_context()
registry = build_source_registry()
print('Trusted tenant:', context.tenant_id)
for source, definition in registry.items():
    print(source.value, definition.model_dump())
assert context.tenant_id == 'northstar'
assert all(definition.read_only for definition in registry.values())

## Part 3 — Routing proposals

The deterministic router supports `KNOWN_ROUTE`, `UNKNOWN`, `AMBIGUOUS`, and `MULTI_ROUTE`. It proposes sources; application policy validates them before retrieval. No router class or model is assumed universally accurate.

In [ ]:
plan = build_retrieval_plan()
for query in plan.queries:
    decision = route_query(query)
    print(query.query_id, decision.route_type.value, [s.value for s in decision.proposed_sources])
dependency_route = route_query(plan.queries[2])
assert dependency_route.route_type == RouteType.MULTI_ROUTE

## Part 4 — Bounded query decomposition

The incident question becomes four evidence questions: current incident, preceding change, dependency/provider, and authorized mitigation. The plan also fixes rewrite, corrective-retrieval, and hop limits.

In [ ]:
print('Mode:', plan.mode.value)
for query in plan.queries:
    print(query.query_id, '—', query.question)
print('Bounds:', {
    'query_rewrites': plan.max_query_rewrites,
    'corrective_retrievals': plan.max_corrective_retrievals,
    'hops': plan.max_hops,
})
assert plan.max_corrective_retrievals == 1

## Part 5 — Initial multi-source retrieval

Retrieve the incident DB and runbook first. The structured incident parameters contain service, region, time window, and limit—but no tenant field. Tenant is bound from trusted context. Do not query the graph until a gap justifies it.

In [ ]:
budget = RetrievalBudget.from_context(context)
bundle = EvidenceBundle(tenant_id=context.tenant_id)
incident_request = build_request(
    plan.queries[0], SourceType.INCIDENT_DB, context, budget,
    structured_parameters=incident_parameters(),
)
runbook_request = build_request(plan.queries[3], SourceType.RUNBOOK_SEARCH, context, budget)
assert incident_request.tenant_id == 'northstar'
initial_results = (
    execute_fixture_retrieval(incident_request),
    execute_fixture_retrieval(runbook_request),
)
bundle = merge_results(bundle, initial_results)
print([item.evidence_id for item in bundle.items])

## Part 6 — Evidence sufficiency

Sufficiency is a typed checkpoint, not an LLM’s feeling. It distinguishes missing incident, dependency, or mitigation evidence, staleness, conflict, and sufficient evidence.

In [ ]:
initial_gap = evaluate_evidence_sufficiency(bundle)
print(initial_gap.model_dump())
assert initial_gap.status == EvidenceStatus.MISSING_DEPENDENCY
assert initial_gap.recommended_source == SourceType.DEPENDENCY_GRAPH

## Part 7 — One bounded corrective retrieval

Only `MISSING_DEPENDENCY` triggers the graph lookup. The controller retrieves once, re-evaluates, and stops when evidence becomes sufficient. The trace records query, route, source, result count, evidence IDs, gap, hop, cost, latency, and stop reason.

In [ ]:
run = run_agentic_controller()
for trace in run.traces:
    print(trace.model_dump(mode='json'))
print('Final gap:', run.gap.status.value)
print('Queries used:', run.budget.used_queries)
assert run.gap.status == EvidenceStatus.SUFFICIENT
assert run.budget.used_queries == 3
assert [t.source_type for t in run.traces].count(SourceType.DEPENDENCY_GRAPH) == 1

## Part 8 — Temporal ranking and conflict handling

The 2026 and 2024 incidents are equally relevant to EU checkout, but the current controlled record outranks the old one. Relevance, authority, and freshness are separate signals. If another credible source asserts a different cause, return `CONFLICT`; do not silently choose one.

In [ ]:
incident_items = [i for i in fixed.bundle.items if i.source_type == SourceType.INCIDENT_DB]
print([(i.evidence_id, i.event_time, i.authority, i.relevance_score) for i in incident_items])
assert incident_items[0].evidence_id == 'incident-eu-2026'
conflict_budget = RetrievalBudget.from_context(context)
conflicted_bundle = add_provider_evidence(run.bundle, context, conflict_budget, conflict=True)
conflict_gap = evaluate_evidence_sufficiency(conflicted_bundle)
print(conflict_gap.model_dump())
assert conflict_gap.status == EvidenceStatus.CONFLICT

## Part 9 — Material claims and evidence links

Claims are explicit artifacts. Every material claim maps to evidence IDs before citations are rendered, so missing support cannot hide behind fluent prose.

In [ ]:
claims = material_claims()
links = claim_links()
for claim in claims:
    linked = next(link.evidence_ids for link in links if link.claim_id == claim.claim_id)
    print(claim.claim_id, claim.text, '->', linked)
assert len(claims) == 3

## Part 10 — Citation verification

A citation URL is not enough. Verification checks evidence existence, tenant, source/version, claim linking, and declared claim support. A generic runbook may be relevant to payments while failing to entail a configuration-change claim.

In [ ]:
citation_report = citation_report_for_run(run)
print(citation_report.model_dump(mode='json'))
assert citation_report.citation_completeness == 1.0
assert citation_report.unsupported_claim_rate == 0.0

runbook = next(item for item in run.bundle.items if item.evidence_id == 'runbook-v7')
claim_2 = claims[1]
unsupported = verify_citations(
    (claim_2,),
    (ClaimEvidenceLink(claim_id='claim-2', evidence_ids=('runbook-v7',)),),
    (Citation(claim_id='claim-2', evidence_id='runbook-v7', source_id=runbook.source_id, source_version=runbook.source_version),),
    run.bundle,
)
assert unsupported.verifications[0].status == ClaimStatus.UNSUPPORTED

## Part 11 — Prompt-injection containment

The malicious result contains potentially useful provider text plus instructions to ignore policy, restart services, and export records. Course 04’s boundary still applies: untrusted content is data. Detection is an observable signal, while deterministic tenant, egress, tool, and approval controls enforce safety.

In [ ]:
malicious = malicious_web_evidence()
before = context.model_dump(mode='json')
inspection = inspect_untrusted_evidence(malicious)
print(inspection)
assert inspection['instructions_authorized'] is False
assert context.model_dump(mode='json') == before
injected_bundle = EvidenceBundle(tenant_id='northstar', items=(malicious, runbook_evidence()))
unsafe_proposal = build_mitigation_proposal(
    'restart-every-production-service', 'production',
    ('web-injected-result', 'runbook-v7'), injected_bundle,
)
assert unsafe_proposal.policy_status == ProposalPolicyStatus.DENIED

## Part 12 — Grounded mitigation, not execution

A mitigation proposal binds action, target, and evidence IDs. Validating the EU provider configuration is an allowed proposal. A rollback stays approval-gated through Course 03. Retrieval evidence never authorizes execution.

In [ ]:
validate_proposal = build_mitigation_proposal(
    'validate-provider-configuration', 'eu-provider',
    ('incident-eu-2026', 'runbook-v7'), run.bundle,
)
rollback_proposal = build_mitigation_proposal(
    'rollback-deployment', 'checkout-eu',
    ('incident-eu-2026', 'runbook-v7'), run.bundle,
)
print(validate_proposal.model_dump(mode='json'))
print(rollback_proposal.model_dump(mode='json'))
assert validate_proposal.policy_status == ProposalPolicyStatus.ALLOWED_PROPOSAL
assert rollback_proposal.policy_status == ProposalPolicyStatus.APPROVAL_REQUIRED

## Part 13 — Safe abstention

If dependency evidence remains unavailable—or the query/hop/cost/time budget blocks the corrective call—the controller returns `INSUFFICIENT_EVIDENCE` with a typed missing-evidence list. It does not fill the gap from model memory.

In [ ]:
abstained = run_agentic_controller(include_dependency=False)
budget_stopped = run_agentic_controller(build_context(max_queries=2))
print(abstained.answer.model_dump(mode='json'))
print(budget_stopped.answer.model_dump(mode='json'))
assert abstained.answer.stop_reason == StopReason.INSUFFICIENT_EVIDENCE
assert abstained.answer.confidence_label == ConfidenceLabel.INSUFFICIENT
assert budget_stopped.answer.stop_reason == StopReason.BUDGET_EXHAUSTED

## Part 14 — Evaluate fixed versus agentic retrieval

These are deterministic fixture measurements, not external benchmark claims. The simple FAQ favors fixed retrieval on cost/latency. The multi-hop incident pays for extra retrieval because required-evidence recall improves. The regression gate also requires zero tenant violations and unsafe action execution.

In [ ]:
comparison = fixed_vs_agentic_comparison()
for row in comparison:
    print(row)
metrics = validate_default_regression_gate()
print(metrics.model_dump())
assert comparison[0]['winner'] == 'fixed'
assert comparison[1]['winner'] == 'agentic'
assert metrics.tenant_violations == 0
assert metrics.unsafe_action_rate == 0

## Part 15 — Optional LangGraph adapter

LangGraph is a widely used open-source option for persistence and conditional orchestration. A production adapter could map `retrieve_initial → evaluate_gap → retrieve_dependency → verify_citations → answer/abstain`. Keep `RetrievalContext`, source validation, budgets, sufficiency, citation verification, and approval outside the adapter so framework choice cannot weaken policy.

In [ ]:
import importlib.util

if importlib.util.find_spec('langgraph'):
    print('LangGraph is available; adapt the tested policy functions as graph nodes.')
else:
    print('LangGraph is optional; the framework-neutral controller is complete.')

## Part 16 — Optional real OpenAI query planner

The offline controller is complete. If you deliberately set `RUN_OPENAI_EXAMPLE=1`, `OPENAI_API_KEY`, and `OPENAI_PLANNER_MODEL`, a model may propose typed evidence questions. Structured output constrains shape only: every route still passes the application-owned source, tenant, web, and budget policy.

In [ ]:
import os
from pydantic import BaseModel, ConfigDict

class QueryProposal(BaseModel):
    model_config = ConfigDict(extra='forbid')
    queries: tuple[RetrievalQuery, ...]

run_live = (
    os.getenv('RUN_OPENAI_EXAMPLE') == '1'
    and os.getenv('OPENAI_API_KEY')
    and os.getenv('OPENAI_PLANNER_MODEL')
)
if run_live:
    from openai import OpenAI
    response = OpenAI().responses.parse(
        model=os.environ['OPENAI_PLANNER_MODEL'],
        input=[
            {'role': 'system', 'content': 'Propose evidence questions only. Never choose tenant, sources, permissions, or budgets.'},
            {'role': 'user', 'content': QUESTION},
        ],
        text_format=QueryProposal,
    )
    for query in response.output_parsed.queries:
        print(query, route_query(query))
else:
    print('Optional OpenAI planner skipped; deterministic lab remains complete.')

## Production upgrade path

| Lab fixture | Production upgrade |
| --- | --- |
| Frozen in-memory context | authenticated tenant/user context in a trusted request boundary |
| Deterministic registry | versioned source catalog with ownership, classification, SLOs, and deny-by-default policy |
| Fixture incident parameters | parameterized read-only queries with enforced tenant predicate and row/time limits |
| In-memory graph | authorized graph service with node/edge schemas and traversal budgets |
| Deterministic sufficiency | calibrated evaluator plus deterministic freshness, scope, and completeness checks |
| Tuple trace | durable, redacted observability with cost, latency, route, gap, and evidence lineage |
| Proposal-only mitigation | Course 03 approval, stable idempotency identity, reconciliation, and audited executor |

Persist policy/source versions, query proposals, admitted requests, evidence handles, gap decisions, claim links, citation results, and stop reasons. Never persist or expose private chain-of-thought.

## Exercises and summary

1. Add one allowed reconciliation retrieval for the conflicting provider fixture, then preserve uncertainty if conflict remains.
2. Add a stale runbook and prove semantic relevance cannot bypass freshness policy.
3. Add a compound route dataset and measure exact, set-based, ambiguous, and unknown route performance.
4. Raise the evidence-recall threshold while holding cost fixed; explain which cases should abstain.
5. Design the durable records needed to resume after initial retrieval without duplicating calls.

**Summary:** Agentic RAG is not ‘search more until the model feels confident.’ Retrieve only for a defined evidence gap. Relevant does not mean supportive; a URL does not make a valid citation. Tenant, source permission, budget, web access, and execution approval are application-owned. More retrieval is not always better. If evidence remains insufficient, abstain.